# 04 · Intent classification demo

Enter an English medical question to obtain a predicted intent and an explanation of the class. This demo reloads the fixed development-stage model without refitting it. SVM decision scores are not probabilities. The project classifies intent and does not generate medical answers. The example questions below are demonstration inputs; they are not experimental results.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "config.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "config.json").exists(), "Open this notebook inside the project."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)
import pandas as pd


In [2]:
from src.predict import predict
question = 'What treatment options are available for asthma?'
predict(question)

{'question': 'What treatment options are available for asthma?',
 'model_id': 'M2',
 'intent': 'Treatment',
 'meaning': 'Asks about treatment or management options.'}

In [3]:
examples = [
    'What is asthma?',
    'What are the symptoms of asthma?',
    'What causes asthma?',
    'How do doctors diagnose asthma?',
    'What are the treatments for asthma?',
    'How can asthma be prevented?'
]
pd.DataFrame([predict(text) for text in examples])

,question,model_id,intent,meaning
0,What is asthma?,M2,Basic information,Asks what the disease or condition is.
1,What are the symptoms of asthma?,M2,Symptoms,Asks about signs or symptoms.
2,What causes asthma?,M2,Causes,Asks why the disease occurs or what causes it.
3,How do doctors diagnose asthma?,M2,Diagnosis and tests,"Asks about diagnosis, examinations or testing."
4,What are the treatments for asthma?,M2,Treatment,Asks about treatment or management options.
5,How can asthma be prevented?,M2,Prevention,Asks how to prevent the disease or reduce its ...


## An actual development-set failure

Retrieve an error from the saved development predictions. The model is not adjusted for this demonstration. High development scores do not establish an ability to understand real patient language.

In [4]:
import json
from src.common import read_questions
model_id = json.loads(Path('models/selected_model.json').read_text())['model_id']
predictions = pd.read_csv('results/dev/predictions.csv')
failed = predictions[(predictions.model_id == model_id) & (predictions.label != predictions.prediction)]
if len(failed):
    example = failed.merge(read_questions()[['question_id', 'question_text']], on='question_id').iloc[0]
    print('True label:', example.label)
    display(predict(example.question_text))
else:
    print('No error in this development split. Do not invent a failure case.')

True label: Basic information


{'question': 'What is (are) How to prevent frostbite and hypothermia ?',
 'model_id': 'M2',
 'intent': 'Prevention',
 'meaning': 'Asks how to prevent the disease or reduce its risk.'}